In [1]:
!pip install -q pyspark

import os, shutil, glob, hashlib
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

In [2]:
from google.colab import files
files.upload()
shutil.move("sales.csv", "data_lake/bronze/sales.csv")

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
bronze_md5 = md5("data_lake/bronze/sales.csv")   # used at the end to prove Bronze is unchanged

Saving sales.csv to sales.csv


In [3]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("DataLake").getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

In [4]:
raw = spark.read.option("header", True).csv("data_lake/bronze/sales.csv")
raw.printSchema()
raw.show(5, truncate=False)
rows_before = raw.count()
print("Total records:", rows_before)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+---

In [5]:
# Duplicates
print("Exact duplicate rows:", rows_before - raw.dropDuplicates().count())

# Nulls / blanks
raw.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
            for c in raw.columns]).show()

# Text problems
text_cols = ["customer_name","product","category","city","state","payment_method","order_status"]
for c in text_cols:
    spaces = raw.filter(F.col(c) != F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")).count()
    variants = raw.select(F.trim(F.col(c))).distinct().count() - raw.select(F.lower(F.trim(F.col(c)))).distinct().count()
    print(f"{c:15} extra-space rows: {spaces:3} | case variants: {variants}")

# Invalid numbers
num = (raw.withColumn("q", F.col("quantity").cast("double"))
          .withColumn("p", F.col("unit_price").cast("double"))
          .withColumn("d", F.col("discount_percent").cast("double")))
print("quantity <= 0:", num.filter(F.col("q") <= 0).count())
print("unit_price <= 0:", num.filter(F.col("p") <= 0).count())
print("discount <0 or >100:", num.filter((F.col("d") < 0) | (F.col("d") > 100)).count())

# Dates
formats = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "dd-MM-yyyy", "MM-dd-yyyy"]
def parse_date(c):
    return F.coalesce(*[F.to_date(F.trim(F.col(c)), f) for f in formats])

raw.withColumn("pattern", F.regexp_replace(F.regexp_replace("order_date", r"\d", "9"), r"[A-Za-z]", "a")) \
   .groupBy("pattern").count().show()                      # shows mixed date formats
bad = raw.filter(F.col("order_date").isNotNull() & parse_date("order_date").isNull())
print("Invalid dates:", bad.count())
bad.select("order_id", "order_date").show()


Exact duplicate rows: 10
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

customer_name   extra-space rows:   3 | case variants: 0
product         extra-space rows:   0 | case variants: 0
category        extra-space rows:   0 | case variants: 5
city            extra-space rows:   3 | case variants: 5
state           extra-space rows:   0 | c

In [6]:
# 1. Remove exact duplicates
df = raw.dropDuplicates()

# 3. Trim, collapse spaces, blank -> null
for c in df.columns:
    df = df.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))
    df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))

# 3. Standardize capitalization (not on product, because it would turn "5L" into "5l")
for c in ["customer_name", "category", "city"]:
    df = df.withColumn(c, F.initcap(F.col(c)))

# 4. Cast numbers, then turn invalid values into null (the row is kept)
df = (df.withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double")))
df = (df.withColumn("quantity", F.when(F.col("quantity") > 0, F.col("quantity")))
        .withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")))
        .withColumn("discount_percent", F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent"))))

# 5. Dates -> one format (yyyy-MM-dd); invalid dates become null
df = df.withColumn("order_date", parse_date("order_date"))

# 2. Missing values
qty_median   = df.approxQuantile("quantity", [0.5], 0.01)[0]
price_median = df.approxQuantile("unit_price", [0.5], 0.01)[0]
prod_med = df.groupBy("product").agg(F.percentile_approx("unit_price", 0.5).alias("prod_price"))
df = (df.join(prod_med, "product", "left")
        .withColumn("unit_price", F.coalesce("unit_price", "prod_price", F.lit(price_median)))
        .drop("prod_price"))
df = df.fillna({"quantity": int(qty_median), "discount_percent": 0.0,
                "customer_name": "Unknown", "payment_method": "Unknown", "city": "Unknown"})

silver = df.select(raw.columns).orderBy("order_id")
rows_after = silver.count()

In [7]:
out = "data_lake/silver/preprocessed_sales"
silver.coalesce(1).write.mode("overwrite").option("header", True).csv(out)
shutil.copy(glob.glob(f"{out}/part-*.csv")[0], "data_lake/silver/sales_silver.csv")

'data_lake/silver/sales_silver.csv'

In [8]:
print("Rows before:", rows_before, "| Rows after:", rows_after)   # expect 1000 -> 990
spark.read.option("header", True).csv(out).show(5)

silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver.columns]).show()
print("Bronze unchanged:", bronze_md5 == md5("data_lake/bronze/sales.csv"))   # must be True

Rows before: 1000 | Rows after: 990
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|    6710.0|            20.0|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|
|  100002|      C0042|   Arjun Singh|      T-Shirt|Clothing|       6|     680.0|             5.0|    Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|
|  100003|      C0348|   Keerthi Rao|       Hoodie|Clothing|       6|    3080.0|            12.0|           UPI|      Pune|   Maharashtra|2025-05-06|

# Preprocessing summary:
- Removed exact duplicate records from the dataset.
- Cleaned whitespace and standardized capitalization across customer_name, category, and city.
- Replaced blank entries with nulls and imputed missing values using the median for quantity and unit_price, 0 for discount, and "Unknown" for text fields.
- Validated quantity, unit_price, and discount_percent; invalid entries were converted to null and subsequently imputed.
- Standardized all valid dates to the yyyy-MM-dd format, while invalid dates were replaced with null values.

In [9]:
files.download("data_lake/silver/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>